# പാഠം 02 - Microsoft Agent ഫ്രെയിംവർക്കിനെ മനസ്സിലാക്കൽ

**Microsoft Agent Framework (MAF)** എന്നു പറയുന്നത് AI എജന്റുകൾ നിർമ്മിക്കാൻ ഉപയോഗിക്കുന്ന ഒരൊറ്റ സമഗ്രമായ ഫ്രെയിംവർക്ക് ആണ്. ഇത് നാല് പ്രധാന ഘടകങ്ങളുള്ള സുതാര്യവും komposable ഉം ആയ ആർക്കിടെക്ചർ നൽകുന്നു:

- **ക്ലയന്റ്** – ഒരു AI മോഡൽ എൻഡ്പോയിന്റുമായി ബന്ധിപ്പിച്ച് ആശയവിനിമയം കൈകാര്യം ചെയ്യുന്നു
- **എജന്റ്** – ക്ലയന്റിനെ നിർദ്ദേശങ്ങളെയും ഉപകരണ നിർവചനങ്ങളുമായിരിച്ച് പാക്ക് ചെയ്യുന്നു
- **ടൂൾസ്** – മോഡൽ വിളിക്കാൻ കഴിയുന്ന കസ്റ്റം ഫംഗ്ഷനുകൾ ഉപയോഗിച്ചുകൊണ്ട് എജന്റിന്റെ ശേഷികളെ വിപുലീകരിക്കുന്നു
- **സെഷൻ** – മൾട്ടി-ടേൺ ഇടപെടലുകൾക്കായി സംഭാഷണ ചരിത്രം നിലനിർത്തുന്നു

ഈ പാഠത്തിൽ, ഞങ്ങൾ ഈ ആശയങ്ങൾ ഉപയോഗിച്ച് ഒരു **യാത്ര ബുക്കിംഗ് എജന്റ്** നിർമ്മിക്കും, അത് ലക്ഷ്യസ്ഥാനത്തിന്റെ ലഭ്യത പരിശോധിക്കും.


## ക്രമീകരണം


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ഏജന്റ് ഫ്രെയിംവർക്ക് ആർക്കിടെക്ചർ മനസിലാക്കൽ

മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഒരു പാളി ആർക്കിടെക്ചർ പിന്തുടരുന്നു:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ക്ലയന്റ്** – ഒരു `FoundryChatClient` ഒരു Azure OpenAI വിന്യാസത്തോടെ കണക്ട് ചെയ്യും. ഇത് സ്ഥിരീകരണം, അപേക്ഷ ഫോർമാറ്റിംഗും മറുപടി പാഴ്സിംഗും കൈകാര്യം ചെയ്യുന്നു.
2. **ഏജന്റ്** – `provider.create_agent()` വഴി ക്ലയന്റിൽ നിന്നും സൃഷ്ടിക്കപ്പെട്ട ഏജന്റ് മോഡൽ ആക്‌സസിനോടു കൂടി നിർദ്ദേശങ്ങൾ (സിസ്റ്റം പ്രാമ്പ്റ്റ്) കൂടാതെ ടൂളുകളും സംയോജിപ്പിക്കുന്നു.
3. **ടൂളുകൾ** – Python ഫംഗ്ഷനുകൾ `@tool` ഉപയോഗിച്ച് അലങ്കരിച്ചിരിക്കുന്നു, ഏജന്റ് അവയെ വിളിച്ച് പ്രവർത്തനങ്ങൾ നടത്തി അല്ലെങ്കിൽ ഡാറ്റ ആന്വേഷിക്കാം.
4. **സെഷൻ** – `agent.create_session()` വഴി സൃഷ്ടിക്കുന്ന ഒരു `AgentSession` ഒബ്‌ജക്റ്റ്, സംഭാഷണ ചരിത്രം സംഭരിക്കുന്നു, മുൻപ് ഉള്ള സാഹചര്യങ്ങൾ ഓർക്കുന്ന മൾട്ടി-ടേൺ സംവാദം സജ്ജമാക്കുന്നു.

ഓരോ പാളിയും ഘട്ടം ഘട്ടമായ് നിർമ്മിക്കാം.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool ഡെക്കറേറ്ററോടെ ടൂളുകൾ ചേർക്കുന്നു

ടൂളുകൾ ഏജന്റുകൾക്ക് വാചകം സൃഷ്ടിക്കുന്നതിന്റെ പുറമേ നടപടി സ്വീകരിക്കാൻ അനുവദിക്കുന്നു. `@tool` ഡെക്കറേറ്റർ ഒരു സാധാരണ Python ഫങ്ഷനെ ഏജന്റ് വിളിക്കാവുന്ന ഒന്നാക്കി മാറ്റുന്നു.

പ്രധാന കാര്യങ്ങൾ:
- ഓരോ പാരാമീറ്ററിനുമുള്ള മോഡൽ മനസ്സിലാക്കുന്നതിനായി `Annotated[type, "description"]` ഉപയോഗിക്കുക.
- ഡോക്സ്ട്രിംഗ് മോഡൽ കാണുന്ന ടൂൾ വിവരണമായി മാറുന്നു.
- `approval_mode="never_require"` എന്ന് സൂചിപ്പിക്കുന്നത് ഉപയോക്തൃ സ്ഥിരീകരണം ഇല്ലാതെ ടൂൾ സ്വയം പ്രവർത്തിക്കും എന്ന് ആണ്.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ഉപകരണങ്ങളോട് ചേർന്നുള്ള ഏജന്റിനെ സൃഷ്ടിക്കൽ

ഇനി നാം ക്ലയന്റ്, നിർദ്ദേശങ്ങൾ, ഉപകരണങ്ങൾ എന്നിവയെ ചേർത്ത് ഒരു ഏജന്റായി മാറ്റുന്നു. `instructions` സിസ്റ്റം പ്രോമ്പ്റ്റിന്റെ মতো പ്രവർത്തിക്കുന്നു — അവ ഏജന്റിന്റെ വ്യക്തിത്വവും പെരുമാറ്റവും ക്രമീകരിക്കുന്നു.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## സെഷനുകളോടെയുള്ള മൾട്ടി-ടേൺ സംഭാഷണങ്ങൾ

ഒരു `AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിക്കപ്പെട്ടത്) ഒരു സംഭാഷണത്തിലെ എല്ലാ സന്ദേശങ്ങളും ട്രാക്ക് ചെയ്യുന്നു. ഓരോ `agent.run()` വിളിക്കും ഒരേ സെഷൻ നൽകുന്നതിലൂടെ, ഏജന്റിന് പൂർണ്ണ സംഭാഷണ ചരിത്രത്തിലേക്ക് പ്രവേശനം ഉണ്ടാകും, മുമ്പത്തെ സന്ദേശങ്ങളെ തിരികെ കാണാൻ കഴിയും.

ഓരോ ടേണിലും ഏജന്ത് നമ്മുടെ ലഭ്യത പരിശോധകനെ(call our availability checker) വിളിക്കാൻ കഴിയുന്നില്ലാതെ `tools=[check_destination_availability]` നാം നൽകുന്നു.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കിന്റെ നാല് പ്രധാന തൂണുകൾ പരിശോധിച്ചു:

| ആശയം | നിങ്ങൾ പഠിച്ചത് |
|---------|------------------|
| **ക്ലയന്റ്** | ക്രെഡൻഷ്യൽ അടിസ്ഥാനമുള്ള ആ.auth ഉപയോഗിച്ച് `FoundryChatClient` Azure OpenAI കണക്റ്റ് ചെയ്യുന്നു |
| **ഏജന്റ്** | `provider.create_agent()` മോഡൽ കണക്ഷൻ നിർദ്ദേശങ്ങളോടും പേരോടെയും ബണ്ട് ചെയ്യുന്നു |
| **ഉപകരണങ്ങൾ** | `@tool` ഡെക്കറേറ്റർ ഏജന്റ് വിളിക്കാനുള്ള Python ഫംഗ്ഷനുകൾ പ്രദർശിപ്പിക്കുന്നു |
| **സെഷൻ** | `agent.create_session()` നിരവധി തവണ സംവാദ ചരിത്രം നിലനിർത്തുന്നു |

ഈ ഘടകങ്ങൾ ചേർത്ത് സ്വാഭാവിക സംഭാഷണങ്ങൾ നടത്താൻ, പുറത്തുള്ള ഫംഗ്ഷനുകൾ വിളിക്കാൻ, കൺടെക്സ്റ്റ് നിലനിർത്താൻ കഴിയുന്ന ഏജന്റുകളെ സൃഷ്ടിക്കുന്നു — പിന്നീട് പാഠങ്ങളിൽ കൂടുതൽ പുരോഗമനമായി ഏജന്റ് മാതൃകകൾക്കുള്ള അടിസ്ഥാനമായി പ്രവർത്തിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
